# `test_api.py` — Backend API Live Test

## Purpose
Sends real HTTP requests to the running ClearView FastAPI backend (`http://localhost:8000`)
and validates every endpoint.

## Prerequisites
1. The backend must be running: `python website/backend/backend_server.py` (or `run_backend.ps1`)
2. `requests` package must be installed: `pip install requests`

## Endpoints tested
| Test | Endpoint | What's verified |
|------|----------|-----------------|
| 1 | `GET /` + `GET /health` | Status 200, `model_loaded`, `device`, `aspects` |
| 2 | `POST /predict` | Single-aspect prediction — colour, smell, packing |
| 3 | Mixed sentiment | colour=positive AND smell=negative for conflict text |
| 4 | `POST /predict_all` or `/analyze` | All-aspects batch endpoint |
| 5 | Edge cases | Short reviews, long reviews |
| 6 | `POST /explain` | XAI endpoint returns `tokens` + `weights` |

In [1]:
from tqdm.auto import tqdm
print("⏳ Starting: Defining function call...")
import time
_start_time = time.time()
import json, time, sys

try:
    import requests
    print('[OK] requests package available')
except ImportError:
    print('[ERROR] requests not installed: pip install requests')
    raise

BASE = 'http://localhost:8000'
SEP  = '=' * 60

def call(label, method, url, **kwargs):
    """Make one HTTP request, print a one-line result, return response or None."""
    try:
        t0  = time.time()
        r   = method(url, timeout=60, **kwargs)
        ms  = (time.time() - t0) * 1000
        tag = '[OK ]' if 200 <= r.status_code < 300 else '[ERR]'
        print(f'{tag} {label:<42}  {r.status_code}  {ms:.0f}ms')
        return r if tag == '[OK ]' else None
    except requests.exceptions.ConnectionError:
        print(f'[CONN] {label} -- cannot reach {BASE}. Is the backend running?')
        return None
    except Exception as exc:
        print(f'[FAIL] {label} -- {exc}')
        return None

print(f'Target: {BASE}')
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Defining function call.")


⏳ Starting: Defining function call...
[OK] requests package available
Target: http://localhost:8000
🕒 Done in 0.08s
✅ Completed: Defining function call.


## Test 1 — Health Check

Verifies the server is up and the model is loaded.

In [2]:
print("⏳ Starting: Loading dependencies and libraries...")
import time
_start_time = time.time()
print(SEP)
print('Test 1: Health Check')
print(SEP)

r = call('GET /', requests.get, f'{BASE}/')
if r:
    d = r.json()
    print(f'  status       : {d.get("status")}')
    print(f'  model_loaded : {d.get("model_loaded")}')
    print(f'  device       : {d.get("device")}')
    print(f'  aspects      : {d.get("aspects")}')

r2 = call('GET /health', requests.get, f'{BASE}/health')
if r2:
    print(f'  /health: {r2.json()}')
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Loading dependencies and libraries.")


⏳ Starting: Loading dependencies and libraries...
Test 1: Health Check
[CONN] GET / -- cannot reach http://localhost:8000. Is the backend running?
[CONN] GET /health -- cannot reach http://localhost:8000. Is the backend running?
🕒 Done in 8.13s
✅ Completed: Loading dependencies and libraries.


## Test 2 — Single-Aspect Prediction

Tests `POST /predict` for three aspects with known expected sentiments.

In [3]:
print("⏳ Starting: Loading dependencies and libraries...")
import time
_start_time = time.time()
print(SEP)
print('Test 2: Single-Aspect Prediction')
print(SEP)

REVIEW = 'I love the colour of this lipstick but the smell is absolutely awful and the packaging broke.'

for aspect, expected in [('colour', 'positive'), ('smell', 'negative'), ('packing', 'negative')]:
    r = call(f'POST /predict  [{aspect}]', requests.post,
             f'{BASE}/predict', json={'text': REVIEW, 'aspect': aspect})
    if r:
        d    = r.json()
        pred = d.get('sentiment') or d.get('label') or d.get('predicted_class') or str(d)[:40]
        conf = d.get('confidence', '?')
        ok   = pred == expected
        print(f'  {aspect:<12} -> {pred:<10} conf={conf}  '
              f'{"CORRECT" if ok else f"WRONG (expected {expected})"}')
        if aspect == 'colour':
            print(f'  Response keys: {list(d.keys())}')
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Loading dependencies and libraries.")


⏳ Starting: Loading dependencies and libraries...
Test 2: Single-Aspect Prediction
[CONN] POST /predict  [colour] -- cannot reach http://localhost:8000. Is the backend running?
[CONN] POST /predict  [smell] -- cannot reach http://localhost:8000. Is the backend running?
[CONN] POST /predict  [packing] -- cannot reach http://localhost:8000. Is the backend running?
🕒 Done in 12.15s
✅ Completed: Loading dependencies and libraries.


## Test 3 — Mixed Sentiment Resolution

Both aspects must be correct for the MSR check to PASS.

In [4]:
print("⏳ Starting: Loading dependencies and libraries...")
import time
_start_time = time.time()
print(SEP)
print('Test 3: Mixed Sentiment Resolution')
print(SEP)

MIXED = 'The colour is beautiful but the smell is absolutely awful.'
results = {}
for aspect, expected in [('colour', 'positive'), ('smell', 'negative')]:
    r = call(f'POST /predict  [{aspect}]', requests.post,
             f'{BASE}/predict', json={'text': MIXED, 'aspect': aspect})
    if r:
        d    = r.json()
        pred = d.get('sentiment') or d.get('label') or '?'
        results[aspect] = pred
        ok = pred == expected
        print(f'  [{"PASS" if ok else "FAIL"}] {aspect} -> {pred}  (expected {expected})')

if len(results) == 2:
    overall = 'PASS' if results.get('colour')=='positive' and results.get('smell')=='negative' else 'FAIL'
    print(f'  Overall MSR result: {overall}')
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Loading dependencies and libraries.")


⏳ Starting: Loading dependencies and libraries...
Test 3: Mixed Sentiment Resolution
[CONN] POST /predict  [colour] -- cannot reach http://localhost:8000. Is the backend running?
[CONN] POST /predict  [smell] -- cannot reach http://localhost:8000. Is the backend running?
🕒 Done in 8.09s
✅ Completed: Loading dependencies and libraries.


## Test 4 — All-Aspects Endpoint

Probes multiple possible endpoint names until one responds successfully.

In [5]:
print("⏳ Starting: Loading dependencies and libraries...")
from tqdm.auto import tqdm
import time
_start_time = time.time()
print(SEP)
print('Test 4: All-Aspects / Analyze Endpoint')
print(SEP)

found = False
for ep in tqdm(['/predict_all', '/analyze', '/analyze_all', '/predict/all'], desc="Processing ep"):
    r = call(f'POST {ep}', requests.post, f'{BASE}{ep}', json={'text': REVIEW})
    if r:
        found  = True
        d      = r.json()
        preds  = d.get('predictions') or d.get('aspects') or (d if isinstance(d, list) else [])
        print(f'  Endpoint       : {ep}')
        print(f'  Aspects returned: {len(preds)}')
        for p in preds:
            name = p.get('name') or p.get('aspect', '?')
            sent = p.get('sentiment') or p.get('label', '?')
            conf = p.get('confidence', '?')
            print(f'    {name:<18} -> {sent:<10} conf={conf}')
        break
if not found:
    print('  [WARN] No all-aspects endpoint found')
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Loading dependencies and libraries.")


⏳ Starting: Loading dependencies and libraries...
Test 4: All-Aspects / Analyze Endpoint


Processing ep:   0%|          | 0/4 [00:00<?, ?it/s]

[CONN] POST /predict_all -- cannot reach http://localhost:8000. Is the backend running?
[CONN] POST /analyze -- cannot reach http://localhost:8000. Is the backend running?
[CONN] POST /analyze_all -- cannot reach http://localhost:8000. Is the backend running?
[CONN] POST /predict/all -- cannot reach http://localhost:8000. Is the backend running?
  [WARN] No all-aspects endpoint found
🕒 Done in 16.28s
✅ Completed: Loading dependencies and libraries.


## Test 5 — Edge Cases

Short inputs (single word) and longer reviews.

In [6]:
print("⏳ Starting: Loading dependencies and libraries...")
import time
_start_time = time.time()
print(SEP)
print('Test 5: Edge Cases')
print(SEP)

edge_cases = [
    ('Short: "Great!"',          {'text': 'Great!',     'aspect': 'colour'}),
    ('Short: "Terrible."',       {'text': 'Terrible.',  'aspect': 'smell'}),
    ('200-char mixed review',    {'text': 'The colour is absolutely stunning. The texture is greasy and heavy. '
                                          'The smell is nice but the price is too high. Shipping was quick '
                                          'but packaging cracked.', 'aspect': 'texture'}),
]
for label, payload in edge_cases:
    r = call(f'POST /predict  [{label}]', requests.post, f'{BASE}/predict', json=payload)
    if r:
        d    = r.json()
        pred = d.get('sentiment') or d.get('label') or '?'
        conf = d.get('confidence', '?')
        print(f'  -> {pred}  conf={conf}')
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Loading dependencies and libraries.")


⏳ Starting: Loading dependencies and libraries...
Test 5: Edge Cases
[CONN] POST /predict  [Short: "Great!"] -- cannot reach http://localhost:8000. Is the backend running?
[CONN] POST /predict  [Short: "Terrible."] -- cannot reach http://localhost:8000. Is the backend running?
[CONN] POST /predict  [200-char mixed review] -- cannot reach http://localhost:8000. Is the backend running?
🕒 Done in 12.19s
✅ Completed: Loading dependencies and libraries.


## Test 6 — Explain / XAI Endpoint

Validates the `/explain` endpoint returns token attribution data.

In [7]:
print("⏳ Starting: Loading dependencies and libraries...")
from tqdm.auto import tqdm
import time
_start_time = time.time()
print(SEP)
print('Test 6: Explain / XAI Endpoint')
print(SEP)

found = False
for ep in tqdm(['/explain', '/explain_attention', '/explain_lime', '/xai'], desc="Processing ep"):
    r = call(f'POST {ep}', requests.post, f'{BASE}{ep}',
             json={'text': REVIEW, 'aspect': 'colour', 'method': 'attention'})
    if r:
        found = True
        d     = r.json()
        print(f'  Endpoint : {ep}')
        print(f'  Keys     : {list(d.keys())}')
        if 'top_tokens' in d:
            print(f'  top_tokens[:3]: {d["top_tokens"][:3]}')
        if 'tokens' in d:
            print(f'  tokens[:5]  : {d["tokens"][:5]}')
        if 'weights' in d:
            print(f'  weights[:5] : {[round(w,3) for w in d["weights"][:5]]}')
        break
if not found:
    print('  [WARN] No explain endpoint found')

print()
print(SEP)
print('API Tests Complete')
print(SEP)
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Loading dependencies and libraries.")


⏳ Starting: Loading dependencies and libraries...
Test 6: Explain / XAI Endpoint


Processing ep:   0%|          | 0/4 [00:00<?, ?it/s]

[CONN] POST /explain -- cannot reach http://localhost:8000. Is the backend running?
[CONN] POST /explain_attention -- cannot reach http://localhost:8000. Is the backend running?
[CONN] POST /explain_lime -- cannot reach http://localhost:8000. Is the backend running?
[CONN] POST /xai -- cannot reach http://localhost:8000. Is the backend running?
  [WARN] No explain endpoint found

API Tests Complete
🕒 Done in 16.21s
✅ Completed: Loading dependencies and libraries.
